## 1. Setup — Connessione a Google Drive ed estrazione dei dati

  Monta Google Drive sull'ambiente Colab e decomprime i due archivi ZIP nella directory locale `/content/tesi`:
  - `subset_manifest.zip` → il CSV con i metadati delle 29.250 immagini del subset
  - `output_subset.zip` → le immagini stesse, organizzate per cluster e categoria

In [1]:
from google.colab import drive
import zipfile
import pathlib

# 1. collego Google Drive
drive.mount('/content/drive')

# 2. definiamo i percorsi vari
# La cartella sorgente su Drive dove sono i due zip
BASE_PATH = pathlib.Path("/content/drive/MyDrive/tesi")

# La cartella di destinazione locale su Colab
DEST = pathlib.Path("/content/tesi")
DEST.mkdir(parents=True, exist_ok=True)

# 3. Elenco dei file da estrarre
zips_to_extract = ["subset_manifest.zip", "output_subset.zip"]

# 4. Ciclo di estrazione
for zip_name in zips_to_extract:
    zip_path = BASE_PATH / zip_name

    if zip_path.exists():
        print(f"Estrazione di {zip_name} in corso...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DEST)
        print(f"Completato: {zip_name}")
    else:
        print(f"Errore: Il file {zip_name} non è stato trovato in {BASE_PATH}")

print("\nTutti i file sono stati estratti in:", DEST)

Mounted at /content/drive
Estrazione di subset_manifest.zip in corso...
Completato: subset_manifest.zip
Estrazione di output_subset.zip in corso...
Completato: output_subset.zip

Tutti i file sono stati estratti in: /content/tesi


## 2. Installazione di CLIP e caricamento del modello

  Installa la libreria OpenAI CLIP direttamente da GitHub, importa le dipendenze e carica il modello **ViT-L/14** sulla GPU (T4 su Colab).

  ViT-L/14 produce embedding di dimensione **768** e rappresenta il miglior trade-off qualità/costo computazionale tra i modelli CLIP disponibili. Il modello viene messo in
  modalità `eval()` per disabilitare dropout e batch norm in training mode.

In [2]:
!pip install git+https://github.com/openai/CLIP.git -q

import clip
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# uso la GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Caricamento del modello ViT-L/14
model, preprocess = clip.load("ViT-L/14", device=device)
model.eval()#modalità inferenza

print(f"Modello caricato: ViT-L/14")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
Device: cuda


100%|████████████████████████████████████████| 890M/890M [00:05<00:00, 175MiB/s]


Modello caricato: ViT-L/14


## 3. Caricamento del manifest e conversione dei percorsi

  Carica il CSV `subset_manifest.csv` (prodotto dalla Phase 5) che contiene i percorsi delle 29.250 immagini del subset.

  La funzione `to_colab_path` converte i percorsi Windows (con backslash, registrati nella Phase 5 su Windows) in percorsi POSIX validi su Colab: estrae la parte del path a
  partire da `output_subset/` e la ricostruisce sotto `/content/tesi/`. Al termine verifica che tutte le immagini siano effettivamente accessibili sul filesystem locale.

In [5]:
SUBSET_ROOT = Path("/content/tesi/output_subset")
MANIFEST = Path("/content/tesi/subset_manifest.csv")
BASE_TESI = SUBSET_ROOT.parent #avrei potuto tranquillamente ridefinire pathlib.Path("/content/drive/MyDrive/tesi"), ma è la stessa cosa

# Caricamento del manifest
df = pd.read_csv(MANIFEST)

def to_colab_path(win_path: str) -> Path:
    # 1. Puliamo il path Windows per isolare la parte da 'output_subset'
    p_str = win_path.replace('/', '\\') # Uniformiamo tutto a backslash
    parts = p_str.split('\\')

    try:
        idx = parts.index("output_subset")
        # Usando il backslash come carattere nel nome
        filename_with_backslashes = "\\".join(parts[idx:])
        return BASE_TESI / filename_with_backslashes
    except ValueError:
        return BASE_TESI / parts[-1]

# Applica e verifica
df["colab_path"] = df["image_path_dest"].apply(to_colab_path)
missing = df["colab_path"].apply(lambda p: not p.exists()).sum()

print(f"Esempio percorso cercato: {df['colab_path'].iloc[0]}")
print(f"Immagini trovate: {len(df) - missing}/{len(df)}")


if missing > 0:
    print("Nota: Controlla che i file siano stati estratti correttamente in:", SUBSET_ROOT)

Esempio percorso cercato: /content/tesi/output_subset\cluster_0\shopping_mall\indoor\00002254.jpg
Immagini trovate: 29250/29250


 ## 4. Estrazione degli embedding CLIP in batch

  Esegue il forward pass del modello su tutte le 29.250 immagini in batch da 256 immagini ciascuno (115 batch totali).

  Per ogni immagine:
  1. **Preprocessing**: resize + crop a 224×224, normalizzazione canali RGB secondo i parametri CLIP
  2. **Encoding**: `model.encode_image()` restituisce un vettore 768D
  3. **Normalizzazione L2**: divide ogni vettore per la sua norma — necessario per usare il dot product come similarità coseno nelle fasi successive

In [6]:
BATCH_SIZE = 256
paths = df["colab_path"].tolist()
N = len(paths)

# Inizializza l'array per gli embeddings (768 è la dimensione per ViT-L/14)
embeddings = np.zeros((N, 768), dtype=np.float32)

for start in tqdm(range(0, N, BATCH_SIZE), desc="Batch"):
    end = min(start + BATCH_SIZE, N)
    batch_paths = paths[start:end]

    imgs = []
    for p in batch_paths:
        try:
            # Caricamento e pre-processing dell'immagine
            img = preprocess(Image.open(p).convert("RGB"))
            imgs.append(img)
        except Exception as e:
            print(f"[WARN] Skip {p}: {e}")
            # Placeholder nero in caso di errore per mantenere l'allineamento degli indici
            imgs.append(torch.zeros(3, 224, 224))

    # Trasformazione in tensor e spostamento sul device (GPU/CPU)
    batch_tensor = torch.stack(imgs).to(device)

    with torch.no_grad():
        # Estrazione delle caratteristiche
        feats = model.encode_image(batch_tensor)
        # Normalizzazione L2 (essenziale per la similarità coseno)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        # Salvataggio nell'array numpy
        embeddings[start:end] = feats.cpu().float().numpy()

print(f"Shape finale: {embeddings.shape}")  # Dovrebbe essere (N, 768)

Batch: 100%|██████████| 115/115 [07:56<00:00,  4.14s/it]

Shape finale: (29250, 768)


## 5. Salvataggio degli embedding e del manifest su Drive

  Salva i risultati direttamente nella cartella `tesi/` su Google Drive:

  - **`clip_image_embeddings.npy`** — matrice numpy `(29250, 768)` float32 con tutti gli embedding L2-normalizzati (~90 MB)
  - **`clip_image_embedding_manifest.csv`** — CSV allineato riga per riga con la matrice, con colonne `cluster_id`, `category`, `image_path_dest`, `embedding_idx` per
  permettere lookup per immagine o per categoria nelle fasi successive

In [7]:
OUT = Path("/content/drive/MyDrive/tesi")

# 1. Salva la matrice degli embeddings in formato binario (.npy)
np.save(OUT / "clip_image_embeddings.npy", embeddings)

# 2. Crea un manifest allineato per i lookup futuri
# Selezioniamo le colonne originali e aggiungiamo l'indice della riga
df_out = df[["cluster_id", "category", "image_path_dest"]].copy()
df_out["embedding_idx"] = df_out.index

# 3. Salva il file CSV su Drive
df_out.to_csv(OUT / "clip_image_embedding_manifest.csv", index=False)

# Output di conferma
print(f"Salvato: clip_image_embeddings.npy ({embeddings.nbytes / 1e6:.1f} MB)")
print(f"Salvato: clip_image_embedding_manifest.csv")

Salvato: clip_image_embeddings.npy (89.9 MB)
Salvato: clip_image_embedding_manifest.csv
